# Sections
Section plots of temperature and salinity compared to observations.

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pyprojroot import here
from ruamel.yaml import YAML

from blomster.grid import read_strait_indices


In [ ]:
# get project root
root_path = Path(here())
yaml = YAML()
with open(root_path.joinpath('notebooks/diagnostics.yml'), "r") as f:
    cfg = yaml.load(f)

# info of diagnostic tool and generated webpages
work_root = cfg['blomster']['work_root']
www_root = cfg['jupyterbook']['www_root']

# info of experiments
caseid = cfg['experiment']['caseid']
input_root = cfg['experiment']['input_root']
grid_label = cfg['experiment']['grid_label']
year_start = cfg['experiment']['year_start']
year_end = cfg['experiment']['year_end']

# info of grids
grid_file = cfg['grids'][grid_label]['grid_file']
section_file = cfg['grids'][grid_label]['section_file']

# info of observational datasets
dataset_name = 'PHC3.0'
obs_file = cfg['datasets'][dataset_name]['ANNUAL_file']

# info of model climatologies
climo_file1 = cfg['climos']['blom']['hy']['ANN']
climo_file2 = cfg['climos']['blom']['hm']['ANN']

print(f"Observational dataset: {obs_file}")


In [ ]:
# set the section names to be plotted
section_names = ['fram_strait']

# mapping of variable names from observations to BLOM, and their colormaps
varmappings = {'temp': 'templvl', 'salt': 'salnlvl'}
cmapmapping = {'temp': cmocean.cm.thermal, 'salt': cmocean.cm.haline}


## Section maps and plots
Location of sections and section plots

In [ ]:
# plot section maps and variables along the sections


def select_section(da, iidx, jidx):
    """Select the values of a 2D grid field along a section of (j, i) indices."""
    return da.isel(
        y=xr.DataArray(jidx, dims="points"),
        x=xr.DataArray(iidx, dims="points"),
    )


def normalize_lon(lon):
    """Convert longitudes to the -180..180 convention."""
    return ((lon + 180) % 360) - 180


def plot_section_map(ax, lon, lat, label=None, color="r"):
    """Plot the points of a section on a polar-stereographic map."""
    ax.plot(
        lon, lat, marker="o", markersize=3, linestyle="-",
        color=color, transform=ccrs.PlateCarree(), label=label,
    )
    ax.set_extent([-180, 180, 60, 90], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.coastlines()
    ax.gridlines(draw_labels=True, linestyle=":", color="gray")
    if label is not None:
        ax.legend(loc="lower left", fontsize=8)
    return ax


# grid, model climatology and observations are opened only once
with xr.open_dataset(grid_file) as grid:
    plat = grid["plat"].load()
    plon = grid["plon"].load()
    pdepth = grid["pdepth"].load()

try:
    climo = xr.open_dataset(climo_file1)
except FileNotFoundError:
    climo = xr.open_dataset(climo_file2)

obs = xr.open_dataset(obs_file)
# observations on the same longitude convention as the model sections
obs = obs.assign_coords(lon=normalize_lon(obs["lon"])).sortby("lon")

for section_name in section_names:
    # read the section indices from the section file
    iidx, jidx, _, _ = read_strait_indices(section_file, section_name)

    # model coordinates and bottom depth along the section
    seclat = select_section(plat, iidx, jidx)
    seclon = select_section(plon, iidx, jidx)
    secdepth = select_section(pdepth, iidx, jidx)
    botdepth = float(secdepth.max())

    # nearest observational grid point for each model point
    lat_sel = xr.DataArray(seclat.values, dims="points")
    lon_sel = xr.DataArray(normalize_lon(seclon).values, dims="points")
    obs_lat = obs["lat"].sel(lat=lat_sel, method="nearest")
    obs_lon = obs["lon"].sel(lon=lon_sel, method="nearest")

    # plot the location of the section points
    fig, axs = plt.subplots(
        1, 2, figsize=(12, 6), subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )
    plot_section_map(axs[0], seclon, seclat, label=section_name, color="r")
    plot_section_map(axs[1], obs_lon, obs_lat, label=section_name, color="b")
    axs[0].set_title("NorESM")
    axs[1].set_title(dataset_name)
    fig.suptitle(f'Section points across "{section_name}"', fontsize=18)
    plt.tight_layout()
    plt.show()

    for varobs, varblom in varmappings.items():
        # model section, with the model longitudes as x coordinate
        section = select_section(climo[varblom].isel(time=0), iidx, jidx)
        section = section.assign_coords(points=seclon)

        # observations at the grid point nearest to each model point
        section_obs = obs[varobs].sel(lat=lat_sel, lon=lon_sel, method="nearest")
        section_obs = section_obs.assign_coords(points=seclon)

        # common colour limits from the 0.1% and 99.9% quantiles
        vmin_model, vmax_model = np.nanquantile(section.values.ravel(), [0.001, 0.999])
        vmin_obs, vmax_obs = np.nanquantile(section_obs.values.ravel(), [0.001, 0.999])
        vmin = min(vmin_model, vmin_obs)
        vmax = max(vmax_model, vmax_obs)

        # simple statistics of the section data
        stats = pd.DataFrame(
            {"min": [vmin_model, vmin_obs], "max": [vmax_model, vmax_obs]},
            index=["NorESM", dataset_name],
        )
        print(f"variable: {varblom}")
        print(stats)

        # plot the sections
        cmap = cmapmapping[varobs]
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 4.5))
        section_obs.plot(
            ax=axs[0], x="points", y="depth", ylim=[0, botdepth],
            vmin=vmin, vmax=vmax, cmap=cmap,
        )
        section.plot(
            ax=axs[1], x="points", y="depth", ylim=[0, botdepth],
            vmin=vmin, vmax=vmax, cmap=cmap,
        )
        axs[0].set_title(f"{dataset_name}: {section_obs.standard_name} across {section_name}")
        axs[1].set_title(f"NorESM: {section.standard_name} across {section_name}")
        axs[0].invert_yaxis()
        axs[1].invert_yaxis()
        plt.show()
